# Imports

In [1]:
import os
import sys
import glob
import tempfile
import pandas as pd
from pcapflower import convert_pcap_to_csv

# Inputs

In [ ]:
DATA_ROOT = "../../../Datasets"

# Folder names to include (relative to DATA_ROOT, e.g. ["benign", "dos_attack"])
# Leave empty to process all folders found under DATA_ROOT
INCLUDE_FOLDERS = []

# Preprocessing

In [ ]:
def find_pcap_folders(root_dir, include=None):
    """Return every folder under root_dir that contains at least one PCAP file.
    If include is non-empty, only folders whose basename is in that set are returned.
    """
    include = set(include) if include else None
    found = []
    for dirpath, _, filenames in os.walk(root_dir):
        if include is not None and os.path.basename(dirpath) not in include:
            continue
        if any(f.endswith((".pcap", ".pcapng")) for f in filenames):
            found.append(dirpath)
    return sorted(found)


def generate_merged_csv_from_pcaps(pcap_dir, output_filename):
    pcap_files = sorted(
        glob.glob(os.path.join(pcap_dir, "*.pcap")) +
        glob.glob(os.path.join(pcap_dir, "*.pcapng"))
    )

    if not pcap_files:
        raise FileNotFoundError(f"No PCAP files found in: {pcap_dir}")

    label = os.path.basename(pcap_dir)
    print(f"[{label}] {len(pcap_files)} PCAP file(s)")

    output_path = os.path.join(pcap_dir, output_filename)
    total_flows = 0
    first_write = True

    with tempfile.TemporaryDirectory(dir=pcap_dir) as tmp_dir:
        for i, pcap_path in enumerate(pcap_files, 1):
            pcap_name = os.path.basename(pcap_path)
            print(f"  [{i}/{len(pcap_files)}] {pcap_name}")

            tmp_csv = os.path.join(tmp_dir, f"{pcap_name}.csv")
            n_flows = convert_pcap_to_csv(pcap_path, tmp_csv, n_jobs= 8)
            print(f"    → {n_flows} flows")

            if n_flows > 0:
                for chunk in pd.read_csv(tmp_csv, chunksize=50_000):
                    chunk["label"] = label
                    chunk.to_csv(output_path, mode="w" if first_write else "a",
                                 index=False, header=first_write)
                    total_flows += len(chunk)
                    first_write = False

    if total_flows == 0:
        raise RuntimeError(f"No flows extracted from: {pcap_dir}")

    print(f"  ✓ {output_path}  ({total_flows} flows)")
    return output_path

In [ ]:
pcap_folders = find_pcap_folders(DATA_ROOT, include=INCLUDE_FOLDERS)
print(f"Found {len(pcap_folders)} folder(s) with PCAPs\n")

for folder_path in pcap_folders:
    folder_name = os.path.basename(folder_path)
    generate_merged_csv_from_pcaps(folder_path, f"merged_{folder_name}.csv")